# Fraud models: what the supplied fraud scores respond to

This notebook trains the two diagnostic classifiers. They explain what drives a fraud score; they are **not** an input to the ranking. The penalty in `fraud_penalty.ipynb` uses the supplied scores directly.

| Part | Model | Data |
|---|---|---|
| A (§2–10) | Consumer tier classifier, per transaction | 34,765 scored user-days |
| B (§11–13) | Merchant tier classifier, per merchant-day | 114 scored merchant-days |

Both parts save their trained models to `data/tables/models/` so `final_findings.ipynb` can load them without retraining.

## Part A: consumer tier classifier

This replaces fraud-probability imputation with a classifier. Each transaction gets a probability of belonging to each of these tiers:

| Tier | Supplied consumer fraud score | Meaning |
|---|---|---|
| 0 | < 30 | scored, low |
| 1 | 30–50 | watch |
| 2 | 50–70 | flagged |
| 3 | ≥ 70 | high |

**Linking scores to transactions.** `consumer_fraud_probability` is recorded per `(user_id, order_date)`, and `order_datetime` has no time of day. So a score can be tied to one exact transaction only when the user made a single transaction that day. Every scored user-day matches at least one transaction, but only about 27% match exactly one. The rest are shared by 2 or more transactions, and nothing in the data says which of them the score refers to.

The classifier is therefore trained per transaction, with each transaction inheriting its day's score. It uses transaction-level features only, so different transactions on the same day can get different predictions. It is evaluated on the single-transaction days, where the transaction-level label is exact, and at the user-day level.

**Scope.** Only scored user-days are used for training. Tier 0 means "scored, but below 30", not "not fraudulent". Scores only exist for a pre-screened subset (the minimum score is about 8%), so predictions on never-scored transactions are outside the training data and should be treated as a ranking for review, not a calibrated probability.

## 1. Setup

In [1]:

# Resolve imports by repository contents, regardless of notebook location.
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scripts" / "process_cleaning.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project repository")
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))


from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window, functions as F
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

# Works when launched from the repository, project2, or notebooks directory.

PROCESSED_PATH = PROJECT_ROOT / "data" / "tables" / "processed"
if not PROCESSED_PATH.is_dir():
    raise FileNotFoundError("Set PROJECT_ROOT to your project2 directory.")
RANDOM_STATE = 42
CUTOFFS = [30, 50, 70]
TIER_NAMES = {0: "<30", 1: "30-50", 2: "50-70", 3: ">=70"}
# Fraud scores exist only in this window (used by the merchant classifier in Part B).
START, END = "2021-02-28", "2022-02-27"
# Without a cap, XGBoost oversubscribes the CPU alongside Spark and slows to a crawl.
XGB_THREADS = 4
spark = SparkSession.builder.config("spark.sql.session.timeZone", "Etc/UTC").config("spark.driver.memory", "6g").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/13 23:27:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Transaction-level behavioural features

These are computed over **all** ~14M transactions and do not use the fraud scores. The fraud score tracks transaction size relative to what is normal for that user and merchant, so the key features are ratios to the user's and the merchant's median transaction.

Aggregates use the whole sample period, including transactions after the one being scored. That's fine for describing the data, but a live system would need them computed from past transactions only.

In [2]:
trans = (
    spark.read.parquet(str(PROCESSED_PATH / "transactions_clean"))
    # Drop the attached scores: they are the label, not features.
    .drop("_source_order", "consumer_fraud_probability", "merchant_fraud_probability")
    .withColumn("order_date", F.to_date("order_datetime"))
)

user_w = Window.partitionBy("user_id")
merchant_w = Window.partitionBy("merchant_abn")
user_merchant_w = Window.partitionBy("user_id", "merchant_abn")
day_w = Window.partitionBy("user_id", "order_date")

trans_features = (
    trans
    .withColumn("user_txn_count", F.count("*").over(user_w))
    .withColumn("user_median_value", F.percentile_approx("dollar_value", 0.5).over(user_w))
    .withColumn("merchant_txn_count", F.count("*").over(merchant_w))
    .withColumn("merchant_median_value", F.percentile_approx("dollar_value", 0.5).over(merchant_w))
    .withColumn("user_merchant_txn_count", F.count("*").over(user_merchant_w))
    # Only used to measure the link between scores and transactions, not as a feature.
    .withColumn("day_txn_count", F.count("*").over(day_w))
)

## 3. Attach scores to transactions

An inner join on `(user_id, order_date)` keeps every transaction on a scored user-day. Each one inherits that day's score.

In [3]:
scores = (
    spark.read.parquet(str(PROCESSED_PATH / "consumer_fraud_probability"))
    .where(F.col("fraud_probability").isNotNull())
    .withColumn("order_date", F.to_date("order_datetime"))
    .drop("order_datetime")
)
merchants = spark.read.parquet(str(PROCESSED_PATH / "merchants_clean")).select(
    "merchant_abn", "revenue_code", "take_rate")
consumers_abs = spark.read.parquet(str(PROCESSED_PATH / "consumers_sa2")).drop(
    "consumer_id", "postcode", "abs_matched")

labelled = (
    scores.join(trans_features, ["user_id", "order_date"], "inner")
    .join(merchants, "merchant_abn", "left")
    .join(consumers_abs, "user_id", "left")
).orderBy("user_id", "order_date", "merchant_abn", "dollar_value").toPandas()

print(f"{scores.count():,} scored user-days -> {len(labelled):,} transactions")

26/10/13 23:27:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


34,765 scored user-days -> 80,348 transactions


In [4]:
p = labelled["fraud_probability"]
labelled["tier"] = np.select([p >= 70, p >= 50, p >= 30], [3, 2, 1], 0)
labelled["order_date"] = pd.to_datetime(labelled["order_date"])
labelled["day_key"] = (labelled["user_id"].astype(str) + "_"
                       + labelled["order_date"].dt.strftime("%Y%m%d"))

days = labelled.drop_duplicates("day_key")
linkage = pd.crosstab(days["tier"],
                      np.where(days["day_txn_count"] == 1, "exact (1 txn)", "shared (2+ txns)"),
                      margins=True).rename(index=TIER_NAMES)
linkage

col_0,exact (1 txn),shared (2+ txns),All
tier,,,
<30,8680,23901,32581
30-50,483,1105,1588
50-70,154,333,487
>=70,40,69,109
All,9357,25408,34765


## 4. Model features

In [5]:
df = labelled.sort_values(["user_id", "order_date", "merchant_abn", "dollar_value"]).reset_index(drop=True)
df["log_value"] = np.log1p(df["dollar_value"])
df["value_vs_user_median"] = df["dollar_value"] / df["user_median_value"]
df["value_vs_merchant_median"] = df["dollar_value"] / df["merchant_median_value"]
df["day_of_week"] = df["order_date"].dt.dayofweek
df["month"] = df["order_date"].dt.month
CAT_COLS = ["revenue_code", "state", "gender"]
for col in CAT_COLS:
    df[col] = df[col].fillna("Missing").astype("category")

ABS_COLS = [c for c in df.columns if c.startswith("abs_")]
FEATURES = ["log_value", "value_vs_user_median", "value_vs_merchant_median",
            "user_txn_count", "user_median_value", "merchant_txn_count",
            "merchant_median_value", "user_merchant_txn_count", "take_rate",
            *CAT_COLS, "day_of_week", "month", *ABS_COLS]

## 5. Grouped, stratified split

The split is by `user_id`, so no consumer appears in both train and test. It is stratified by tier, so the test set gets its share of the rare ≥70 days.

In [6]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(sgkf.split(df, df["tier"], groups=df["user_id"]))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
assert not set(train["user_id"]) & set(test["user_id"])

pd.DataFrame({
    "train days": train.drop_duplicates("day_key")["tier"].value_counts(),
    "test days": test.drop_duplicates("day_key")["tier"].value_counts(),
}).rename(index=TIER_NAMES).sort_index()

,train days,test days
tier,,
30-50,1299,289
50-70,384,103
<30,26084,6497
>=70,87,22


## 6. Cumulative models: P(score ≥ 30), P(≥ 50), P(≥ 70)

In [7]:
# Every transaction on a scored day inherits that day's score, so labels on
# shared days are noisy. Weighting by 1/day_txn_count makes each user-day
# contribute a total weight of 1, however many transactions it had. This beat
# training on single-transaction days only, especially at the user-day level.
train_weight = 1.0 / train["day_txn_count"]

models = {}
for k in CUTOFFS:
    models[k] = XGBClassifier(
        n_estimators=400, learning_rate=0.03, max_depth=4, subsample=0.8,
        colsample_bytree=0.8, min_child_weight=5, enable_categorical=True,
        tree_method="hist", n_jobs=XGB_THREADS, random_state=RANDOM_STATE)
    models[k].fit(train[FEATURES], (train["fraud_probability"] >= k).astype(int),
                  sample_weight=train_weight)

In [8]:
def predict_cumulative(frame):
    probs = pd.DataFrame({k: models[k].predict_proba(frame[FEATURES])[:, 1] for k in CUTOFFS},
                         index=frame.index)
    # Separate models can cross; enforce P(>=70) <= P(>=50) <= P(>=30).
    probs[50] = np.minimum(probs[50], probs[30])
    probs[70] = np.minimum(probs[70], probs[50])
    return probs

def tier_probabilities(cum):
    return pd.DataFrame({"p_tier_0": 1 - cum[30], "p_tier_1": cum[30] - cum[50],
                         "p_tier_2": cum[50] - cum[70], "p_tier_3": cum[70]})

cum = predict_cumulative(test)

for k in CUTOFFS:
    test[f"p_ge_{k}"] = cum[k]
test = test.join(tier_probabilities(cum))

## 7. Evaluation

Each cutoff is scored on two sets:

- **Single-transaction days**: the transaction-level label is exact.
- **All user-days**: a day's prediction is its highest-scoring transaction.

The PR-AUC of a random ranking equals the prevalence, which is the baseline to compare against.

In [9]:
day_pred = test.groupby("day_key").agg(
    fraud_probability=("fraud_probability", "first"),
    **{f"p_ge_{k}": (f"p_ge_{k}", "max") for k in CUTOFFS})
single = test[test["day_txn_count"] == 1]

rows = []
for k in CUTOFFS:
    for name, frame in [("single-txn days", single), ("all user-days (max)", day_pred)]:
        y = (frame["fraud_probability"] >= k).astype(int)
        rows.append({"cutoff": f">={k}", "evaluated on": name, "n": len(frame),
                     "positives": int(y.sum()), "prevalence": y.mean(),
                     "PR-AUC": average_precision_score(y, frame[f"p_ge_{k}"]),
                     "ROC-AUC": roc_auc_score(y, frame[f"p_ge_{k}"]),
                     "mean predicted": frame[f"p_ge_{k}"].mean()})
pd.DataFrame(rows).round(3)

,cutoff,evaluated on,n,positives,prevalence,PR-AUC,ROC-AUC,mean predicted
0,>=30,single-txn days,1853,135,0.073,0.971,0.997,0.074
1,>=30,all user-days (max),6911,414,0.060,0.940,0.986,0.101
2,>=50,single-txn days,1853,44,0.024,0.916,0.997,0.020
3,>=50,all user-days (max),6911,125,0.018,0.913,0.994,0.028
4,>=70,single-txn days,1853,10,0.005,0.764,0.998,0.003
5,>=70,all user-days (max),6911,22,0.003,0.558,0.988,0.005


The tier confusion matrix uses single-transaction days. A transaction is assigned the highest tier whose cumulative probability is ≥ 0.5. That 0.5 is a default operating point, not a tuned one.

In [10]:
def assign_tier(frame, threshold=0.5):
    return np.select([frame["p_ge_70"] >= threshold, frame["p_ge_50"] >= threshold,
                      frame["p_ge_30"] >= threshold], [3, 2, 1], 0)

cm = confusion_matrix(single["tier"], assign_tier(single), labels=[0, 1, 2, 3])
pd.DataFrame(cm, index=[f"actual {TIER_NAMES[t]}" for t in range(4)],
             columns=[f"pred {TIER_NAMES[t]}" for t in range(4)])

,pred <30,pred 30-50,pred 50-70,pred >=70
actual <30,1703,15,0,0
actual 30-50,10,78,3,0
actual 50-70,0,11,22,1
actual >=70,0,0,6,4


## 8. Which transaction on a shared day?

For flagged days with 2 or more transactions, this compares the highest-scoring transaction in the day with the second highest. A large gap means the model singles out one transaction as the likely source of the day's score.

In [11]:
multi = test[(test["day_txn_count"] > 1) & (test["tier"] >= 1)]
ranked = multi.sort_values("p_ge_30", ascending=False).groupby("day_key")["p_ge_30"]
attribution = pd.DataFrame({"top": ranked.nth(0).values, "second": ranked.nth(1).values})
print(f"{len(attribution):,} flagged test days with 2+ transactions")
print(attribution.describe().round(3))
print("Share with top >= 0.5 and second < 0.2:",
      round(((attribution["top"] >= 0.5) & (attribution["second"] < 0.2)).mean(), 3))

279 flagged test days with 2+ transactions
           top   second
count  279.000  279.000
mean     0.814    0.060
std      0.269    0.016
min      0.044    0.001
25%      0.787    0.049
50%      0.946    0.059
75%      0.982    0.070
max      0.996    0.123
Share with top >= 0.5 and second < 0.2: 0.853


## 9. Feature importance (≥ 50 model, gain)

In [12]:
gain = pd.Series(models[50].get_booster().get_score(importance_type="gain")).sort_values()
gain.tail(12).plot.barh(figsize=(7, 4), title="XGBoost gain, P(score >= 50)")
plt.tight_layout()
plt.show()

/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27273/2072979229.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Save the models

The three classifiers and the held-out test transactions are written to `data/tables/models/`, so `final_findings.ipynb` can load the trained weights instead of retraining.

In [13]:
# Save the trained classifiers so other notebooks can reuse them without retraining.
MODEL_DIR = PROJECT_ROOT / "data" / "tables" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for k in CUTOFFS:
    models[k].save_model(MODEL_DIR / f"consumer_fraud_ge{k}.json")

# The held-out test transactions, with the features and category dtypes the models were
# trained on, so the weights can be applied without rebuilding the 14M-row window features.
KEEP = ["user_id", "day_key", "day_txn_count", "dollar_value", "fraud_probability", "tier"]
test[KEEP + FEATURES].to_parquet(MODEL_DIR / "consumer_fraud_test.parquet", index=False)

print(f"written to {MODEL_DIR}:")
for path in sorted(MODEL_DIR.glob("consumer_fraud_*")):
    print(f"  {path.name}")

written to /Users/matthewwihartanto/Documents/University /Year 3/Sem 1/Applied DS/GitHub/project2/data/tables/models:
  consumer_fraud_ge30.json
  consumer_fraud_ge50.json
  consumer_fraud_ge70.json
  consumer_fraud_test.parquet


### Consumer classifier: interpretation and limitations

- **What the score tracks.** On single-transaction days, the supplied score has a Spearman correlation of about 0.91 with log transaction value. The median transaction overall is about $62, but the median largest transaction on a scored day is about $3,300. The classifier is mostly learning "large relative to this user's and this merchant's usual transaction".
- **Linking to transactions.** The link is exact only on single-transaction days (about 27% of scored days). On shared days, the model's per-transaction scores point to a likely transaction, but that's the model's inference, not something recorded in the data.
- **The ≥70 tier is thin.** There are 109 user-days in total and about 20 in the test fold, so its metrics are noisy. Use cross-validation or several seeds before quoting them.
- **Scope.** Tier probabilities are only meaningful for transactions like the scored ones. On never-scored transactions, use them as a ranking for review, not as a calibrated fraud probability.
- **Operating points.** The 0.5 cutoff in the confusion matrix is a default. Pick flag thresholds from the precision each tier needs.
- Only the trained models and the held-out test transactions are exported (§10). No per-transaction predictions feed the ranking.

## Part B: merchant tier classifier

The same 30 / 50 / 70 cutoffs, applied to the 114 scored merchant-days. The predictors describe how unusual the day was for that merchant: revenue, count, average and largest transaction relative to the merchant's usual. This part tests how far the small merchant sample can go.

## 11. Merchant-day features

Every merchant-day in the scoring window is compared with that merchant's usual (median) behaviour over the same window.

In [14]:
trans_window = (
    spark.read.parquet(str(PROCESSED_PATH / "transactions_clean"))
    .withColumn("order_date", F.to_date("order_datetime"))
    .where(F.col("order_date").between(START, END))
)
merchant_w = Window.partitionBy("merchant_abn")
merchant_txn = trans_window.groupBy("merchant_abn").agg(
    F.percentile_approx("dollar_value", 0.5).alias("merchant_txn_median"),
    F.mean("dollar_value").alias("merchant_txn_mean"),
    F.stddev("dollar_value").alias("merchant_txn_std"))
merchant_day = (
    trans_window.groupBy("merchant_abn", "order_date")
    .agg(F.sum("dollar_value").alias("revenue"), F.count("*").alias("txn_count"),
         F.mean("dollar_value").alias("avg_value"), F.max("dollar_value").alias("max_value"))
    .withColumn("typical_revenue", F.percentile_approx("revenue", 0.5).over(merchant_w))
    .withColumn("typical_count", F.percentile_approx("txn_count", 0.5).over(merchant_w))
    .withColumn("revenue_mean", F.mean("revenue").over(merchant_w))
    .withColumn("revenue_std", F.stddev("revenue").over(merchant_w))
    .withColumn("active_days", F.count("*").over(merchant_w))
    .join(merchant_txn, "merchant_abn")
)

merchant_scores = (spark.read.parquet(str(PROCESSED_PATH / "merchant_fraud_probability"))
                   .where(F.col("fraud_probability").isNotNull())
                   .withColumn("order_date", F.to_date("order_datetime")).drop("order_datetime"))
days = merchant_scores.join(merchant_day, ["merchant_abn", "order_date"], "inner").orderBy("merchant_abn", "order_date").toPandas()
assert len(days) == merchant_scores.count(), "every scored merchant-day should match transactions"

days["revenue_ratio"] = days["revenue"] / days["typical_revenue"]
days["count_ratio"] = days["txn_count"] / days["typical_count"]
days["avg_value_ratio"] = days["avg_value"] / days["merchant_txn_median"]
days["max_value_ratio"] = days["max_value"] / days["merchant_txn_median"]
# Share of the day's revenue that came from its single largest transaction.
days["max_share_of_day"] = days["max_value"] / days["revenue"]
# A merchant with a single active day has no spread; treat it as "not unusual".
days["revenue_z"] = ((days["revenue"] - days["revenue_mean"]) / days["revenue_std"]).fillna(0)
days["max_value_z"] = ((days["max_value"] - days["merchant_txn_mean"])
                       / days["merchant_txn_std"]).fillna(0)
days["merchant_txn_cv"] = (days["merchant_txn_std"] / days["merchant_txn_mean"]).fillna(0)
days["log_revenue"] = np.log(days["revenue"])
days["log_typical_revenue"] = np.log(days["typical_revenue"])
days["log_active_days"] = np.log(days["active_days"])
p = days["fraud_probability"]
days["tier"] = np.select([p >= 70, p >= 50, p >= 30], [3, 2, 1], 0)

MERCHANT_FEATURES = ["revenue_ratio", "count_ratio", "avg_value_ratio", "max_value_ratio",
            "max_share_of_day", "revenue_z", "max_value_z", "merchant_txn_cv",
            "log_revenue", "log_typical_revenue", "log_active_days"]
print(f"{len(days)} scored merchant-days across {days['merchant_abn'].nunique()} merchants")
days.groupby("tier")[MERCHANT_FEATURES].median().rename(index=TIER_NAMES).T.round(2) \
    .assign(**{"Spearman with score": [spearmanr(days[c], p).statistic for c in MERCHANT_FEATURES]}).round(2)

114 scored merchant-days across 61 merchants


tier,<30,30-50,50-70,>=70,Spearman with score
revenue_ratio,3.03,4.72,10.44,5.35,0.72
count_ratio,2.74,3.00,2.00,1.00,0.03
avg_value_ratio,1.35,2.12,3.39,5.35,0.71
max_value_ratio,3.62,4.08,4.87,5.67,0.39
max_share_of_day,0.34,0.45,0.61,1.00,0.27
revenue_z,2.58,3.62,6.18,3.86,0.38
max_value_z,2.45,2.73,3.44,4.15,0.35
merchant_txn_cv,0.65,0.73,0.88,0.83,0.29
log_revenue,10.69,10.90,8.22,10.74,-0.00
log_typical_revenue,9.57,9.47,5.85,9.21,-0.55


- **Higher tiers are single oversized transactions.** Moving up the tiers, the day's count stays near normal, while the average and largest transaction grow relative to the merchant's usual.
- **At ≥ 70, one transaction is the whole day.** The median `max_share_of_day` is 1.0.
- **This matches the consumer side.** There, the score tracked transaction size relative to the user's usual.

## 12. Cumulative classifiers, leave-one-merchant-out

With only 114 rows, one test split would be too noisy. Instead each merchant is held out in turn: it is scored by a model trained on the other 60 merchants.

Two model types are compared:
- **Logistic regression** (L2-regularised, standardised features)
- **Shallow XGBoost** (depth 2)

The PR-AUC of a random ranking equals the prevalence.

In [15]:
MODELS = {
    "logistic": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=5000)),
    "xgboost (depth 2)": lambda: XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=2, subsample=0.8,
        n_jobs=XGB_THREADS, random_state=RANDOM_STATE),
}
groups = days["merchant_abn"]
oof, rows = {}, []
for k in CUTOFFS:
    y = (days["fraud_probability"] >= k).astype(int)
    for name, make in MODELS.items():
        pred = cross_val_predict(make(), days[MERCHANT_FEATURES], y, groups=groups,
                                 cv=LeaveOneGroupOut(), method="predict_proba")[:, 1]
        oof[(k, name)] = pred
        rows.append({"cutoff": f">={k}", "model": name, "positive days": int(y.sum()),
                     "positive merchants": days.loc[y == 1, "merchant_abn"].nunique(),
                     "prevalence": y.mean(), "PR-AUC": average_precision_score(y, pred),
                     "ROC-AUC": roc_auc_score(y, pred)})
pd.DataFrame(rows).round(3)

/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountere

/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountere

/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountere

,cutoff,model,positive days,positive merchants,prevalence,PR-AUC,ROC-AUC
0,>=30,logistic,73,46,0.640,0.960,0.929
1,>=30,xgboost (depth 2),73,46,0.640,0.976,0.954
2,>=50,logistic,27,25,0.237,0.933,0.962
3,>=50,xgboost (depth 2),27,25,0.237,0.911,0.949
4,>=70,logistic,7,7,0.061,0.388,0.833
5,>=70,xgboost (depth 2),7,7,0.061,0.580,0.903


In [16]:
coefs = pd.DataFrame({
    f">={k}": MODELS["logistic"]().fit(days[MERCHANT_FEATURES], (days["fraud_probability"] >= k).astype(int))[-1].coef_[0]
    for k in CUTOFFS}, index=MERCHANT_FEATURES)
coefs.round(2).sort_values(">=50")

/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountere

,>=30,>=50,>=70
log_revenue,0.69,-0.88,0.29
log_typical_revenue,-0.45,-0.71,0.02
merchant_txn_cv,0.04,-0.52,-0.07
max_value_ratio,-0.43,-0.24,-0.10
count_ratio,0.22,-0.15,0.13
log_active_days,0.29,0.16,-0.44
revenue_ratio,1.87,0.18,0.20
max_value_z,0.04,0.35,0.81
revenue_z,0.81,0.48,0.47
avg_value_ratio,1.30,0.88,0.05


- **≥ 30 and ≥ 50 are learnable.** Both are well above the random baseline on merchants the model never saw.
- **≥ 70 is not.** There are 7 days, each from a different merchant, so every fold predicts a merchant from 6 examples. Its numbers are unstable and shouldn't be reported as a model result.
- **Scope.** The model is trained only on scored days. Like the consumer classifier, it ranks days that were already selected for scoring. It does not give a fraud probability for an ordinary day.

The penalty in `fraud_penalty.ipynb` uses the **observed** scores directly rather than these predictions. Unscored merchants are covered by the `suspect` flag built there.

## 13. Save the merchant models

Written to `data/tables/models/` alongside the consumer models.

In [17]:
# Save the merchant tier classifier so other notebooks can reuse it without retraining.
MODEL_DIR = PROJECT_ROOT / "data" / "tables" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
XGB_NAME = "xgboost (depth 2)"

# The leave-one-merchant-out fits above are 61 separate models. The saved model is one
# final fit on all 114 scored days, so it has seen every merchant: use the out-of-fold
# columns below, not this model's own predictions, to judge how well it separates days.
for k in CUTOFFS:
    final_model = MODELS[XGB_NAME]().fit(days[MERCHANT_FEATURES], (days["fraud_probability"] >= k).astype(int))
    final_model.save_model(MODEL_DIR / f"merchant_fraud_ge{k}.json")

export = days[["merchant_abn", "order_date", "fraud_probability", "tier",
               "revenue", "typical_revenue", "max_value"]
              + [c for c in MERCHANT_FEATURES if c not in ("revenue", "typical_revenue", "max_value")]].copy()
for k in CUTOFFS:
    export[f"p_ge_{k}_oof"] = oof[(k, XGB_NAME)]
export.to_parquet(MODEL_DIR / "merchant_fraud_days.parquet", index=False)

print(f"written to {MODEL_DIR}:")
for path in sorted(MODEL_DIR.glob("merchant_fraud_*")):
    print(f"  {path.name}")

written to /Users/matthewwihartanto/Documents/University /Year 3/Sem 1/Applied DS/GitHub/project2/data/tables/models:
  merchant_fraud_days.parquet
  merchant_fraud_ge30.json
  merchant_fraud_ge50.json
  merchant_fraud_ge70.json


### Merchant classifier: interpretation and limitations

- Leave-one-merchant-out PR-AUC:
  - ≥ 30: about 0.96, against a random baseline of 0.64.
  - ≥ 50: about 0.91–0.93, against 0.24.
  - ≥ 70: not usable. There are only 7 days, each from a different merchant.
- The score follows how unusual the day is for that merchant, especially a single transaction far above the merchant's usual. This mirrors the consumer score.
- With 114 rows the saved model is a final fit on every scored day. Judge it by the leave-one-merchant-out results above, not by its own predictions on those days.